<!--BOOK_INFORMATION-->
<img align="left" style="padding-right:10px;" src="figures/PDSH-cover-small.png">
*This notebook contains an excerpt from the [Python Data Science Handbook](http://shop.oreilly.com/product/0636920034919.do) by Jake VanderPlas; the content is available [on GitHub](https://github.com/jakevdp/PythonDataScienceHandbook).*

*The text is released under the [CC-BY-NC-ND license](https://creativecommons.org/licenses/by-nc-nd/3.0/us/legalcode), and code is released under the [MIT license](https://opensource.org/licenses/MIT). If you find this content useful, please consider supporting the work by [buying the book](http://shop.oreilly.com/product/0636920034919.do)!*

<!--NAVIGATION-->
< [Working with Time Series](03.11-Working-with-Time-Series.ipynb) | [Contents](Index.ipynb) | [Further Resources](03.13-Further-Resources.ipynb) >

# 3.13 High-Performance Pandas / 高性能Pandas: eval() and query()

As we've already seen in previous sections, the power of the PyData stack is built upon the ability of NumPy and Pandas to push basic operations into C via an intuitive syntax: examples are vectorized/broadcasted operations in NumPy, and grouping-type operations in Pandas.
While these abstractions are efficient and effective for many common use cases, they often rely on the creation of temporary intermediate objects, which can cause undue overhead in computational time and memory use.

🐍 Python数据科学生态环境的强大力量建立在 NumPy与Pandas 的基础之上，并通过直观的语法将基本操作转换成 C语言：

🐍 在 NumPy 里是向量化 / 广播运算，在 Pandas 里是分组型的运算

🐍 虽然这些抽象功能可以简洁高效地解决许多问题，但是它们经常需要创建临时 中间对象 ，这样就会占用大量的计算时间与内存

As of version 0.13 (released January 2014), Pandas includes some experimental tools that allow you to directly access C-speed operations without costly allocation of intermediate arrays.
These are the ``eval()`` and ``query()`` functions, which rely on the [Numexpr](https://github.com/pydata/numexpr) package.
In this notebook we will walk through their use and give some rules-of-thumb about when you might think about using them.

🐍 实验性工具，让用户可以直接运行 C语言 速度的操作，不需要十分费力地配置中间数组。它们就是eval() 和query() 函数，

🐍 都依赖于Numexpr (https://github.com/pydata/numexpr) 程序包

## Motivating ``query()`` and ``eval()``: Compound Expressions

We've seen previously that NumPy and Pandas support fast vectorized operations; for example, when adding the elements of two arrays:

In [1]:
import numpy as np
rng = np.random.RandomState(42)
x = rng.rand(1000000)
y = rng.rand(1000000)
%timeit x + y

3 ms ± 60.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


As discussed in [Computation on NumPy Arrays: Universal Functions](02.03-Computation-on-arrays-ufuncs.ipynb), this is much faster than doing the addition via a Python loop or comprehension:

🐍 向量化运算比普通的 Python 循环或列表综合要快很多

In [2]:
%timeit np.fromiter((xi + yi for xi, yi in zip(x, y)), dtype=x.dtype, count=len(x))

190 ms ± 9.35 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


But this abstraction can become less efficient when computing compound expressions.
For example, consider the following expression:

🐍 这种运算在处理复合代数式（compound expression）问题时的效率比较低

In [3]:
mask = (x > 0.5) & (y < 0.5)

Because NumPy evaluates each subexpression, this is roughly equivalent to the following:

In [4]:
tmp1 = (x > 0.5)
tmp2 = (y < 0.5)
mask = tmp1 & tmp2

In other words, *every intermediate step is explicitly allocated in memory*. If the ``x`` and ``y`` arrays are very large, this can lead to significant memory and computational overhead.
The Numexpr library gives you the ability to compute this type of compound expression element by element, without the need to allocate full intermediate arrays.
The [Numexpr documentation](https://github.com/pydata/numexpr) has more details, but for the time being it is sufficient to say that the library accepts a *string* giving the NumPy-style expression you'd like to compute:

🐍 Numexpr程序库：其实就是用一个 NumPy风格的字符串代数式进行运算，不为中间过程分配全部内存的前提下，完成元素到元素的复合代数式运算。

In [5]:
import numexpr
mask_numexpr = numexpr.evaluate('(x > 0.5) & (y < 0.5)')
np.allclose(mask, mask_numexpr)

True

The benefit here is that Numexpr evaluates the expression in a way that does not use full-sized temporary arrays, and thus can be much more efficient than NumPy, especially for large arrays.
The Pandas ``eval()`` and ``query()`` tools that we will discuss here are conceptually similar, and depend on the Numexpr package.

🐍 Pandas 的 eval() 和 query() 工具其实也是基于 Numexpr 实现的。

## 3.13.2 ``pandas.eval()`` for Efficient Operations / 实现高性能运算

The ``eval()`` function in Pandas uses string expressions to efficiently compute operations using ``DataFrame``s.
For example, consider the following ``DataFrame``s:

In [6]:
import pandas as pd
nrows, ncols = 100000, 100
rng = np.random.RandomState(42)
df1, df2, df3, df4 = (pd.DataFrame(rng.rand(nrows, ncols))
                      for i in range(4))

To compute the sum of all four ``DataFrame``s using the typical Pandas approach, we can just write the sum:

In [7]:
# DataFrame 字符串代数式 求和

%timeit df1 + df2 + df3 + df4

59.7 ms ± 2.74 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


The same result can be computed via ``pd.eval`` by constructing the expression as a string:

In [8]:
# pd.eval() 函数

%timeit pd.eval('df1 + df2 + df3 + df4')

24.6 ms ± 1.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


The ``eval()`` version of this expression is about 50% faster (and uses much less memory), while giving the same result:

🐍 这个 eval() 版本的代数式比普通方法快一倍（而且内存消耗更少），

In [9]:
# 🐍 结果一致：np.allclose()

np.allclose(df1 + df2 + df3 + df4,
            pd.eval('df1 + df2 + df3 + df4'))

True

### Operations supported by ``pd.eval()`` / 支持的运算

As of Pandas v0.16, ``pd.eval()`` supports a wide range of operations.
To demonstrate these, we'll use the following integer ``DataFrame``s:

In [10]:
df1, df2, df3, df4, df5 = (pd.DataFrame(rng.randint(0, 1000, (100, 3)))
                           for i in range(5))

#### (1) Arithmetic operators / 算术运算符
``pd.eval()`` supports all arithmetic operators. For example:

In [11]:
# 🐍 加 减 乘 除, pd.eval()支持所有的算术运算

result1 = -df1 * df2 / (df3 + df4) - df5
result2 = pd.eval('-df1 * df2 / (df3 + df4) - df5')
np.allclose(result1, result2)

True

#### (2) Comparison operators / 比较运算符
``pd.eval()`` supports all comparison operators, including chained expressions:

In [12]:
# 🐍 pd.eval()支持所有的比较运算符，包括链式代数式

result1 = (df1 < df2) & (df2 <= df3) & (df3 != df4)
result2 = pd.eval('df1 < df2 <= df3 != df4')
np.allclose(result1, result2)

True

#### (3) Bitwise operators / 位运算符
``pd.eval()`` supports the ``&`` and ``|`` bitwise operators:

In [13]:
# 🐍 pd.eval()支持 &（与）和|（或）等位运算符

result1 = (df1 < 0.5) & (df2 < 0.5) | (df3 < df4)
result2 = pd.eval('(df1 < 0.5) & (df2 < 0.5) | (df3 < df4)')
np.allclose(result1, result2)

True

In addition, it supports the use of the literal ``and`` and ``or`` in Boolean expressions:

In [14]:
# 🐍 pd.eval()支持 布尔类型的代数式中使用and 和or 等字面值

result3 = pd.eval('(df1 < 0.5) and (df2 < 0.5) or (df3 < df4)')
np.allclose(result1, result3)

True

#### (4) Object attributes and indices / 对象属性与索引

``pd.eval()`` supports access to object attributes via the ``obj.attr`` syntax, and indexes via the ``obj[index]`` syntax:

In [20]:
# 🐍 pd.eval() 可以通过 obj.attr 语法获取对象属性，通过 obj[index] 语法获取对象索引

result1 = df2.T[0] + df3.iloc[1]
result2 = pd.eval('df2.T[0] + df3.iloc[1]')
np.allclose(result1, result2)

True

#### (5) Other operations / 其他运算
Other operations such as function calls, conditional statements, loops, and other more involved constructs are currently *not* implemented in ``pd.eval()``.

🐍 目前pd.eval() 还不支持函数调用、条件语句、循环以及更复杂的运算

If you'd like to execute these more complicated types of expressions, you can use the Numexpr library itself.

## 3.13.3 ``DataFrame.eval()`` for Column-Wise Operations / 实现列间运算

Just as Pandas has a top-level ``pd.eval()`` function, ``DataFrame``s have an ``eval()`` method that works in similar ways.

🐍 pd.eval() 是Pandas 的顶层函数，因此 DataFrame 有一个 eval() 方法可以做类似的运
算。

The benefit of the ``eval()`` method is that columns can be referred to *by name*.
We'll use this labeled array as an example:

In [15]:
import pandas as pd
df = pd.DataFrame(rng.rand(1000, 3), columns=['A', 'B', 'C'])
df.head()

,A,B,C
0,0.375506,0.406939,0.069938
1,0.069087,0.235615,0.154374
2,0.677945,0.433839,0.652324
3,0.264038,0.808055,0.347197
4,0.589161,0.252418,0.557789


Using ``pd.eval()`` as above, we can compute expressions with the three columns like this:

In [16]:
# 🐍 使用 pd.eval() 方法的好处是可以借助列名称进行运算

result1 = (df['A'] + df['B']) / (df['C'] - 1)
result2 = pd.eval("(df.A + df.B) / (df.C - 1)")
np.allclose(result1, result2)

True

In [ ]:
result2 = pd.eval("(df['A'] + df['B']) / (df['C'] - 1)")
result2

In [18]:
result1 = (df.A + df.B) / (df.C - 1)
result1

0      -0.841283
1      -0.360327
2      -3.197758
3      -1.642291
4      -1.903118
         ...    
995    -0.213265
996    -3.273031
997   -85.440864
998    -1.984602
999    -1.560814
Length: 1000, dtype: float64

The ``DataFrame.eval()`` method allows much more succinct evaluation of expressions with the columns:

In [19]:
# 🐍 DataFrame.eval() 方法可以通过列名称实现简洁的代数式

result3 = df.eval('(A + B) / (C - 1)')
np.allclose(result1, result3)

True

Notice here that we treat *column names as variables* within the evaluated expression, and the result is what we would wish.

### 1. Assignment in DataFrame.eval() / 用DataFrame.eval()新增列

In addition to the options just discussed, ``DataFrame.eval()``  also allows assignment to any column.
Let's use the ``DataFrame`` from before, which has columns ``'A'``, ``'B'``, and ``'C'``:

In [20]:
df.head()

,A,B,C
0,0.375506,0.406939,0.069938
1,0.069087,0.235615,0.154374
2,0.677945,0.433839,0.652324
3,0.264038,0.808055,0.347197
4,0.589161,0.252418,0.557789


We can use ``df.eval()`` to create a new column ``'D'`` and assign to it a value computed from the other columns:

In [21]:
# 🐍 用 df.eval() 创建一个新的列 'D'

df.eval('D = (A + B) / C', inplace=True)
df.head()

,A,B,C,D
0,0.375506,0.406939,0.069938,11.187620
1,0.069087,0.235615,0.154374,1.973796
2,0.677945,0.433839,0.652324,1.704344
3,0.264038,0.808055,0.347197,3.087857
4,0.589161,0.252418,0.557789,1.508776


In the same way, any existing column can be modified:

In [22]:
# 🐍 用 df.eval() 修改已有的列 'D'

df.eval('D = (A - B) / C', inplace=True)
df.head()

,A,B,C,D
0,0.375506,0.406939,0.069938,-0.449425
1,0.069087,0.235615,0.154374,-1.078728
2,0.677945,0.433839,0.652324,0.374209
3,0.264038,0.808055,0.347197,-1.566886
4,0.589161,0.252418,0.557789,0.603708


### 2. Local variables in DataFrame.eval() / DataFrame.eval()使用局部变量

The ``DataFrame.eval()`` method supports an additional syntax that lets it work with local Python variables.
Consider the following:

In [23]:
# 🐍 DataFrame.eval() 方法支持通过 @ 符号使用 Python 的局部变量

column_mean = df.mean(1)
result1 = df['A'] + column_mean
result2 = df.eval('A + @column_mean')
np.allclose(result1, result2)

True

The ``@`` character here marks a *variable name* rather than a *column name*, and lets you efficiently evaluate expressions involving the two "namespaces": the namespace of columns, and the namespace of Python objects.
Notice that this ``@`` character is only supported by the ``DataFrame.eval()`` *method*, not by the ``pandas.eval()`` *function*, because the ``pandas.eval()`` function only has access to the one (Python) namespace.

🐍 @ 符号表示“这是一个变量名称而不是一个列名称”，从而让你灵活地用两个“命名空
间”的资源（列名称的命名空间和Python 对象的命名空间）计算代数式。

🐍 需要注意的是，@ 符号只能在DataFrame.eval() 方法中使用，而不能在pandas.eval() 函数中使用，因为pandas.eval() 函数只能获取一个（Python）命名空间的内容。

## 3.13.4 DataFrame.query() Method / DataFrame.query()方法

The ``DataFrame`` has another method based on evaluated strings, called the ``query()`` method.
Consider the following:

In [24]:
# pd.eval(): 用 DataFrame 列创建的代数式

result1 = df[(df.A < 0.5) & (df.B < 0.5)]
result2 = pd.eval('df[(df.A < 0.5) & (df.B < 0.5)]')
np.allclose(result1, result2)

True

As with the example used in our discussion of ``DataFrame.eval()``, this is an expression involving columns of the ``DataFrame``.
It cannot be expressed using the ``DataFrame.eval()`` syntax, however!
Instead, for this type of filtering operation, you can use the ``query()`` method:

In [25]:
# 🐍 用 DateFrame.query() 方法可以处理这种过滤运算

result2 = df.query('A < 0.5 and B < 0.5')
np.allclose(result1, result2)

True

In addition to being a more efficient computation, compared to the masking expression this is much easier to read and understand.
Note that the ``query()`` method also accepts the ``@`` flag to mark local variables:

🐍 除了计算性能更优之外，DateFrame.query()方法的语法也比掩码代数式语法更好理解

In [26]:
Cmean = df['C'].mean()
result1 = df[(df.A < Cmean) & (df.B < Cmean)]
result2 = df.query('A < @Cmean and B < @Cmean')
np.allclose(result1, result2)

True

## 3.13.5 Performance: When to Use These Functions / 性能决定使用时机

When considering whether to use these functions, there are two considerations: *computation time* and *memory use*.
Memory use is the most predictable aspect. As already mentioned, every compound expression involving NumPy arrays or Pandas ``DataFrame``s will result in implicit creation of temporary arrays:
For example, this:

🐍 考虑要不要用这两个函数时, 需要思考两个方面：计算时间和内存消耗，而内存消耗是更重要的影响因素

In [27]:
x = df[(df.A < 0.5) & (df.B < 0.5)]

Is roughly equivalent to this:

In [29]:
tmp1 = df.A < 0.5
tmp2 = df.B < 0.5
tmp3 = tmp1 & tmp2
x = df[tmp3]

If the size of the temporary ``DataFrame``s is significant compared to your available system memory (typically several gigabytes) then it's a good idea to use an ``eval()`` or ``query()`` expression.
You can check the approximate size of your array in bytes using this:

In [30]:
df.values.nbytes

32000

On the performance side, ``eval()`` can be faster even when you are not maxing-out your system memory.
The issue is how your temporary ``DataFrame``s compare to the size of the L1 or L2 CPU cache on your system (typically a few megabytes in 2016); if they are much bigger, then ``eval()`` can avoid some potentially slow movement of values between the different memory caches.
In practice, I find that the difference in computation time between the traditional methods and the ``eval``/``query`` method is usually not significant–if anything, the traditional method is faster for smaller arrays!
The benefit of ``eval``/``query`` is mainly in the saved memory, and the sometimes cleaner syntax they offer.

🐍 普通的计算方法与eval/ query 计算方法在计算时间上的差异并非总是那么明显，普通方法在处理较小的数组时反而速度更快！

🐍 eval/ query 方法的优点主要是节省内存，有时语法也更加简洁。

We've covered most of the details of ``eval()`` and ``query()`` here; for more information on these, you can refer to the Pandas documentation.
In particular, different parsers and engines can be specified for running these queries; for details on this, see the discussion within the ["Enhancing Performance" section](http://pandas.pydata.org/pandas-docs/dev/enhancingperf.html).

<!--NAVIGATION-->
< [Working with Time Series](03.11-Working-with-Time-Series.ipynb) | [Contents](Index.ipynb) | [Further Resources](03.13-Further-Resources.ipynb) >